In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.data import load_dataset, split_data
from src.models import *
from src.evaluate import *
!pip install torch -q
import torch
print(torch.__version__)

THRESH = {"telco": 0.36, "bank": 0.32}


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


2.15.0+cpu


In [2]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

X_train, X_test, y_train, y_test = split_data(load_dataset("telco"))
P = X_train.drop(columns=["tenure", "TotalCharges"])
D = pd.get_dummies(P, drop_first=True).astype(float)
D["T"] = X_train["tenure"].clip(lower=0.5).values
D["E"] = y_train.values

cph = CoxPHFitter(penalizer=0.1).fit(D, duration_col="T", event_col="E")
res = proportional_hazard_test(cph, D, time_transform="rank")
t = res.summary.sort_values("p")
print(t.head(12).round(4))
print("\nfeatures with p < 0.05:", int((t.p < 0.05).sum()), "of", len(t))

                                       test_statistic       p  -log2(p)
Contract_Two year                             19.9209  0.0000   16.9187
MultipleLines_Yes                             12.6560  0.0004   11.3833
StreamingMovies_Yes                            9.2692  0.0023    8.7452
StreamingTV_Yes                                9.1927  0.0024    8.6850
PaymentMethod_Mailed check                     7.2184  0.0072    7.1146
Contract_One year                              5.6492  0.0175    5.8395
PaymentMethod_Credit card (automatic)          5.3023  0.0213    5.5532
Partner_Yes                                    5.2530  0.0219    5.5123
DeviceProtection_Yes                           4.3129  0.0378    4.7245
TechSupport_Yes                                3.5188  0.0607    4.0428
OnlineSecurity_Yes                             1.9223  0.1656    2.5942
OnlineBackup_Yes                               1.7578  0.1849    2.4352

features with p < 0.05: 9 of 21


In [3]:
def group_report(name, col):
    Xtr, _, ytr, _ = split_data(load_dataset(name))
    proba = pd.read_csv(f"../results/oof_xgb_{name}.csv")["proba"].values
    d = pd.DataFrame({"g": Xtr[col].values, "y": ytr.values, "flag": proba >= THRESH[name]})
    out = d.groupby("g").apply(lambda s: pd.Series({
        "n": len(s),
        "churn_rate": s.y.mean(),
        "flag_rate": s.flag.mean(),
        "recall": s.flag[s.y == 1].mean(),
        "false_alarm_rate": s.flag[s.y == 0].mean()}), include_groups=False).round(3)
    print(f"\n{name} by {col}"); print(out)

group_report("bank", "Gender")
group_report("bank", "Geography")
group_report("telco", "gender")
group_report("telco", "SeniorCitizen")


bank by Gender
             n  churn_rate  flag_rate  recall  false_alarm_rate
g                                                              
Female  3613.0       0.251      0.271   0.671             0.137
Male    4387.0       0.165      0.157   0.591             0.071

bank by Geography
              n  churn_rate  flag_rate  recall  false_alarm_rate
g                                                               
France   3993.0       0.159      0.143   0.557             0.065
Germany  2005.0       0.331      0.394   0.754             0.215
Spain    2002.0       0.166      0.152   0.548             0.074

telco by gender
             n  churn_rate  flag_rate  recall  false_alarm_rate
g                                                              
Female  2801.0       0.266      0.316   0.690             0.180
Male    2833.0       0.264      0.314   0.688             0.179

telco by SeniorCitizen
        n  churn_rate  flag_rate  recall  false_alarm_rate
g                           

In [4]:
import shap
from lime.lime_tabular import LimeTabularExplainer

def lime_vs_shap(name, n=100, k=5, seed=42):
    Xtr, Xte, ytr, yte = split_data(load_dataset(name))
    pipe = make_xgb(Xtr).fit(Xtr, ytr)
    prep, mdl = pipe.named_steps["prep"], pipe.named_steps["model"]
    Z = prep.transform(Xtr).astype("float64")
    names = list(prep.get_feature_names_out())

    proba = pipe.predict_proba(Xtr)[:, 1]
    flagged = np.where(proba >= THRESH[name])[0]
    pick = np.random.RandomState(seed).choice(flagged, size=min(n, len(flagged)), replace=False)

    sv = shap.TreeExplainer(mdl).shap_values(Z[pick])
    lime = LimeTabularExplainer(Z, feature_names=names, mode="classification",
                                discretize_continuous=False, random_state=seed)

    overlaps = []
    for i, row in enumerate(pick):
        e = lime.explain_instance(Z[row], mdl.predict_proba, num_features=k, num_samples=2000)
        lime_top = {names[j] for j, _ in e.as_map()[1]}
        shap_top = set(np.array(names)[np.argsort(-np.abs(sv[i]))[:k]])
        overlaps.append(len(lime_top & shap_top) / k)
    print(f"{name}: mean top-{k} overlap LIME vs SHAP over {len(pick)} flagged customers =",
          round(np.mean(overlaps), 3), "| share with overlap >= 0.6:",
          round(np.mean(np.array(overlaps) >= 0.6), 3))

lime_vs_shap("telco")
lime_vs_shap("bank")

telco: mean top-5 overlap LIME vs SHAP over 100 flagged customers = 0.662 | share with overlap >= 0.6: 0.89
bank: mean top-5 overlap LIME vs SHAP over 100 flagged customers = 0.732 | share with overlap >= 0.6: 0.96


In [5]:
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_censored
from src.survival import to_surv, predictors

X_train, X_test, y_train, y_test = split_data(load_dataset("telco"))
S_train, P_train = to_surv(X_train, y_train), predictors(X_train)

def fit_deepsurv(Xtr, Str, epochs=200, seed=42):
    torch.manual_seed(seed)
    order = np.argsort(-Str["time"])                      # longest tenure first, so a running sum covers each risk set
    x = torch.tensor(Xtr[order], dtype=torch.float32)
    e = torch.tensor(Str["event"][order].astype(np.float32))
    net = nn.Sequential(nn.Linear(x.shape[1], 32), nn.ReLU(), nn.Dropout(0.1),
                        nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 1))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-3)
    for _ in range(epochs):
        net.train(); opt.zero_grad()
        r = net(x).squeeze(1)                              # risk score per customer
        loss = -((r - torch.logcumsumexp(r, 0)) * e).sum() / e.sum()   # Cox negative partial log-likelihood
        loss.backward(); opt.step()
    net.eval()
    return net

scores = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(P_train, S_train["event"]):
    prep = make_preprocessor(P_train)
    Xtr, Xva = prep.fit_transform(P_train.iloc[tr]), prep.transform(P_train.iloc[va])
    net = fit_deepsurv(Xtr, S_train[tr])
    with torch.no_grad():
        risk = net(torch.tensor(Xva, dtype=torch.float32)).squeeze(1).numpy()
    Sva = S_train[va]
    scores.append(concordance_index_censored(Sva["event"], Sva["time"], risk)[0])

print("DeepSurv C-index (mean, sd):", round(np.mean(scores), 3), round(np.std(scores), 3))
print("Cox 0.867, XGBoost-AFT 0.865, RSF 0.852 for comparison")

DeepSurv C-index (mean, sd): 0.868 0.004
Cox 0.867, XGBoost-AFT 0.865, RSF 0.852 for comparison


In [6]:
from sksurv.metrics import integrated_brier_score

TIMES = np.arange(3, 61, 3)          # same months as the other models

def breslow_baseline(risk, S):
    """Cumulative baseline hazard at each event time (Breslow estimator)."""
    order = np.argsort(S["time"])
    t, e, r = S["time"][order], S["event"][order], np.exp(risk[order])
    at_risk = np.cumsum(r[::-1])[::-1]               # sum of risk over everyone still at risk
    ut = np.unique(t[e])
    H, cum = [], 0.0
    for u in ut:
        d = ((t == u) & e).sum()
        cum += d / at_risk[np.searchsorted(t, u, side="left")]
        H.append(cum)
    return ut, np.array(H)

ibs = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(P_train, S_train["event"]):
    prep = make_preprocessor(P_train)
    Xtr, Xva = prep.fit_transform(P_train.iloc[tr]), prep.transform(P_train.iloc[va])
    Str, Sva = S_train[tr], S_train[va]
    net = fit_deepsurv(Xtr, Str)
    with torch.no_grad():
        r_tr = net(torch.tensor(Xtr, dtype=torch.float32)).squeeze(1).numpy()
        r_va = net(torch.tensor(Xva, dtype=torch.float32)).squeeze(1).numpy()
    ut, H = breslow_baseline(r_tr, Str)
    idx = np.searchsorted(ut, TIMES, side="right") - 1
    H0 = np.where(idx >= 0, H[np.clip(idx, 0, None)], 0.0)
    est = np.exp(-np.outer(np.exp(r_va), H0))        # predicted survival at each month
    ibs.append(integrated_brier_score(Str, Sva, est, TIMES))

print("DeepSurv integrated Brier score (mean, sd):", round(np.mean(ibs), 3), round(np.std(ibs), 3))
print("Cox 0.084, XGBoost-AFT 0.082, RSF 0.092, no-features 0.175 for comparison")

DeepSurv integrated Brier score (mean, sd): 0.083 0.003
Cox 0.084, XGBoost-AFT 0.082, RSF 0.092, no-features 0.175 for comparison


In [7]:
from sklearn.metrics import average_precision_score
from src.evaluate import oof_proba

SENS = {"telco": ["gender", "SeniorCitizen"], "bank": ["Gender", "Geography"]}
GROUPCOL = {"telco": ["gender", "SeniorCitizen"], "bank": ["Gender", "Geography"]}
COSTS = {"telco": dict(offer=78, V=777.0), "bank": dict(offer=100, V=1000.0)}
S_RATE = 0.3

def avg_cost(y, flag, offer, V):
    flagged = offer + np.where(y, (1 - S_RATE) * V, 0)
    unflagged = np.where(y, V, 0)
    return float(np.mean(np.where(flag, flagged, unflagged)))

def best_threshold(y, proba, offer, V):
    grid = np.arange(0.05, 0.96, 0.01)
    costs = [avg_cost(y, proba >= t, offer, V) for t in grid]
    i = int(np.argmin(costs))
    return round(float(grid[i]), 2), costs[i]

rows, probas = [], {}
for name in ["telco", "bank"]:
    Xtr, _, ytr, _ = split_data(load_dataset(name))
    y = ytr.values.astype(bool)
    for label, X in [("with sensitive", Xtr), ("without sensitive", Xtr.drop(columns=SENS[name]))]:
        proba = oof_proba(make_xgb(X), X, ytr)
        probas[(name, label)] = proba
        thr, cost = best_threshold(y, proba, **COSTS[name])
        rows.append({"dataset": name, "model": label, "PR-AUC": round(average_precision_score(y, proba), 3),
                     "best_threshold": thr, "cost": round(cost, 1),
                     "nobody": round(avg_cost(y, np.zeros(len(y), bool), **COSTS[name]), 1)})
print(pd.DataFrame(rows).to_string(index=False))

dataset             model  PR-AUC  best_threshold  cost  nobody
  telco    with sensitive   0.663            0.36 188.1   206.2
  telco without sensitive   0.660            0.32 188.3   206.2
   bank    with sensitive   0.701            0.32 185.7   203.8
   bank without sensitive   0.665            0.33 187.5   203.8


In [8]:
# group error rates with and without the sensitive attributes, each at its own cost-optimal threshold
for name in ["telco", "bank"]:
    Xtr, _, ytr, _ = split_data(load_dataset(name))
    y = ytr.values.astype(bool)
    for col in GROUPCOL[name]:
        out = []
        for label in ["with sensitive", "without sensitive"]:
            proba = probas[(name, label)]
            thr, _ = best_threshold(y, proba, **COSTS[name])
            d = pd.DataFrame({"g": Xtr[col].values, "y": y, "flag": proba >= thr})
            r = d.groupby("g").apply(lambda s: pd.Series({
                "flag_rate": s.flag.mean(), "recall": s.flag[s.y].mean(),
                "false_alarm": s.flag[~s.y].mean()}), include_groups=False).round(3)
            r.columns = pd.MultiIndex.from_product([[label], r.columns])
            out.append(r)
        print(f"\n{name} by {col}"); print(pd.concat(out, axis=1))


telco by gender
       with sensitive                    without sensitive                   
            flag_rate recall false_alarm         flag_rate recall false_alarm
g                                                                            
Female          0.316  0.690       0.180             0.357  0.739       0.218
Male            0.314  0.688       0.179             0.352  0.733       0.214

telco by SeniorCitizen
  with sensitive                    without sensitive                   
       flag_rate recall false_alarm         flag_rate recall false_alarm
g                                                                       
0          0.272  0.648       0.155             0.315  0.708       0.193
1          0.536  0.810       0.345             0.553  0.817       0.369

bank by Gender
       with sensitive                    without sensitive                   
            flag_rate recall false_alarm         flag_rate recall false_alarm
g                               